# 02 — Indices: NBR, NDVI, dNBR, FIREMON severity

Inputs: the outputs of `01_data` (run it first in the same Colab runtime).
Outputs: `data/dnbr_2023_2025.tif`, `data/severity_2023_2025.tif`, `figures/severity_map.png`

In [ ]:
![ -d /content/jasper-burn-severity ] || git clone https://github.com/MegaDeadCowboy/jasper-burn-severity.git /content/jasper-burn-severity
%cd /content/jasper-burn-severity
!pip install -q rioxarray geopandas

In [ ]:
import sys; sys.path.insert(0, ".")
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, xarray as xr, rioxarray
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from src.indices import ndvi, nbr, dnbr, classify_severity, SEVERITY_LABELS
from google.colab import drive; drive.mount("/content/drive")

DATA = Path("/content/drive/MyDrive/jasper-burn-severity/data"); DATA.mkdir(parents=True, exist_ok=True)
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
UTM, PIX_HA = 32611, 20 * 20 / 10_000          # 0.04 ha per 20 m pixel
BANDS = ["B04", "B08", "B8A", "B12"]           # band order written by 01_data
VEG_T = 0.10                                   # pre-fire NDVI mask (Decision Log 2026-10-08)
for f in ["s2_pre_2023.tif", "s2_post_2025.tif", "nbac_jasper_2024.gpkg"]:
    assert (DATA / f).exists(), f"missing {f}: run 01_data first in this runtime"

In [ ]:
def load(path):
    da = rioxarray.open_rasterio(path, masked=True).assign_coords(band=BANDS)
    return da.to_dataset("band")

pre, post = load(DATA / "s2_pre_2023.tif"), load(DATA / "s2_post_2025.tif")
perim = gpd.read_file(DATA / "nbac_jasper_2024.gpkg").to_crs(UTM)
print(pre.rio.crs, pre.rio.resolution(), dict(pre.sizes))

## 1. Indices

In [ ]:
nbr_pre, nbr_post = nbr(pre), nbr(post)
ndvi_pre, ndvi_post = ndvi(pre), ndvi(post)
d = dnbr(nbr_pre, nbr_post).rio.write_crs(UTM)

inside = d.rio.clip(perim.geometry, drop=False)
ring = perim.buffer(2000).difference(perim.unary_union)            # 2 km unburned margin
outside = d.rio.clip(ring.geometry, drop=False)
for name, a in [("inside perimeter", inside), ("2 km ring outside", outside)]:
    v = a.values[np.isfinite(a.values)]
    print(f"dNBR {name}: n={v.size:,}  median={np.median(v):.3f}  p5={np.percentile(v,5):.3f}  p95={np.percentile(v,95):.3f}")

The ring median is the **dNBR offset**, which reflects phenology and sensor differences between the two years over unburned ground. It is reported here but **not** applied: the FIREMON thresholds are used as-is. Applying an offset is a method decision for the brief's Decision Log.

## 2. FIREMON severity classes

In [ ]:
sev = classify_severity(d).rio.write_crs(UTM)
veg = ndvi_pre >= VEG_T
sev_in = sev.where(veg).rio.clip(perim.geometry, drop=False)
sev_all_in = sev.rio.clip(perim.geometry, drop=False)              # float, NaN outside perimeter
nonveg_ha = float((np.isfinite(sev_all_in) & ~veg).sum()) * PIX_HA
print(f"Masked as non-vegetated (NDVI < {VEG_T}) inside perimeter: {nonveg_ha:,.1f} ha")

counts = pd.Series(sev_in.values.ravel()).dropna().astype(int).value_counts().sort_index()
area = pd.DataFrame({"class": [SEVERITY_LABELS[i] for i in counts.index],
                     "pixels": counts.values,
                     "area_ha": (counts.values * PIX_HA).round(1)})
area["pct"] = (100 * area.area_ha / area.area_ha.sum()).round(1)
print(f"Classified area inside NBAC perimeter: {area.area_ha.sum():,.1f} ha (NBAC POLY_HA = {perim.POLY_HA.sum():,.1f})")
area

Areas cover pixels with pre-fire NDVI ≥ 0.10 only; rock, ice and water are masked (Decision Log 2026-10-08). The townsite is not yet masked: it still needs a verified boundary source, and that choice will be logged when it is made.

## 3. Severity map

In [ ]:
# Ordinal one-hue ramp (Low -> High), validated with the dataviz validator (--ordinal, light surface)
SEV_COLORS = ["#f0a060", "#e07434", "#bd4f14", "#7f2c06"]
cmap = ListedColormap(SEV_COLORS)
ext = [float(d.x.min()), float(d.x.max()), float(d.y.min()), float(d.y.max())]

base = post["B8A"]
base = (base / base.quantile(0.98)).clip(0, 1)
burned = sev.where((sev >= 1) & veg) - 1                           # 0..3 -> Low..High; unburned + non-veg transparent

fig, ax = plt.subplots(figsize=(8, 9), dpi=200)
ax.imshow(base, cmap="gray", vmin=0, vmax=1.4, extent=ext)          # vmax>1 keeps the basemap recessive
ax.imshow(burned, cmap=cmap, vmin=-0.5, vmax=3.5, extent=ext, interpolation="nearest")
perim.boundary.plot(ax=ax, color="#383835", lw=0.6)
ranges = ["0.10–0.27", "0.27–0.44", "0.44–0.66", "> 0.66"]
ax.legend(handles=[Patch(color=c, label=f"{l}  (dNBR {r})") for c, l, r in zip(SEV_COLORS, SEVERITY_LABELS[1:], ranges)]
          + [Patch(facecolor="none", edgecolor="#383835", label="NBAC 2024 perimeter")],
          loc="lower left", frameon=True, framealpha=0.9, fontsize=8, title="Burn severity (FIREMON)", title_fontsize=8)
ax.set_title(f"2024 Jasper Wildfire: burn severity, Sentinel-2 dNBR (Jul–Aug 2023 vs Jul–Aug 2025), pre-fire NDVI ≥ {VEG_T}", fontsize=9, loc="left")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(FIG / "severity_map.png", bbox_inches="tight")

## 4. Save

In [ ]:
d.rio.to_raster(DATA / "dnbr_2023_2025.tif", compress="deflate")
sev.fillna(255).astype("uint8").rio.write_nodata(255).rio.to_raster(DATA / "severity_2023_2025.tif", compress="deflate")
area.to_csv(FIG / "severity_area_inside_perimeter_veg.csv", index=False)
print("saved")